In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm 
tqdm.pandas()

pd.set_option("display.max.colwidth", None)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("originals files/himym_full_transcripts.csv")

df.head()

,episode,name,line
0,4x02 - The Best Burger in New York,Barney,"Goliath National Bank. The world leader in credit and banking. God, I love Goliath National Bank!\n"
1,4x02 - The Best Burger in New York,Ted,"Okay, first of all, you look like the last pick in the draft. And, second, why are you so excited about some bank?\n"
2,4x02 - The Best Burger in New York,Barney,"Our company just bought them out in a ruthless takeover. Took two months. Cost 2,000 jobs. It was brutal. Who wants a T-shirt? Hey, Marshall, they're hiring in the legal department. I could get you a job.\n"
3,4x02 - The Best Burger in New York,Lily,"Barney, Marshall didn't quit his last soul-sucking corporate job just to go work at a bank. He's gonna be an environmental lawyer.\n"
4,4x02 - The Best Burger in New York,Ted,"So, what do you guys want to do for dinner?\n"


In [4]:
df.shape

(24010, 3)

In [5]:
model_id = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

emotion_analyzer = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 46149.21it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
emotion_analyzer(df.iloc[0,2])

[[{'label': 'joy', 'score': 0.8631492853164673},
  {'label': 'neutral', 'score': 0.05854996666312218},
  {'label': 'disgust', 'score': 0.02795964851975441},
  {'label': 'anger', 'score': 0.023999027907848358},
  {'label': 'surprise', 'score': 0.01640997640788555},
  {'label': 'sadness', 'score': 0.005888327956199646},
  {'label': 'fear', 'score': 0.0040437751449644566}]]

In [7]:
emotion_analyzer(df.iloc[0,2])[0]

[{'label': 'joy', 'score': 0.8631492853164673},
 {'label': 'neutral', 'score': 0.05854996666312218},
 {'label': 'disgust', 'score': 0.02795964851975441},
 {'label': 'anger', 'score': 0.023999027907848358},
 {'label': 'surprise', 'score': 0.01640997640788555},
 {'label': 'sadness', 'score': 0.005888327956199646},
 {'label': 'fear', 'score': 0.0040437751449644566}]

In [8]:
def analisis_emociones(texto):
    try:
        resultado = emotion_analyzer(texto)[0][0]
        return {
            "emotion": resultado["label"],
            "score": resultado["score"]
        }
    except:
        return "no datos"

In [9]:
analisis_emociones(df.iloc[0,2])

{'emotion': 'joy', 'score': 0.8631492853164673}

In [10]:
df[["emotion", "score"]] = df["line"].progress_apply(analisis_emociones).apply(pd.Series)

100%|██████████| 24010/24010 [08:13<00:00, 48.65it/s]


In [11]:
df.head(2)

,episode,name,line,emotion,score
0,4x02 - The Best Burger in New York,Barney,"Goliath National Bank. The world leader in credit and banking. God, I love Goliath National Bank!\n",joy,0.863149
1,4x02 - The Best Burger in New York,Ted,"Okay, first of all, you look like the last pick in the draft. And, second, why are you so excited about some bank?\n",surprise,0.763355


In [12]:
df["emotion"].value_counts()

emotion
neutral     10911
surprise     4492
disgust      2466
anger        2325
joy          1977
sadness      1277
fear          562
Name: count, dtype: int64

In [13]:
df[df["emotion"] == "neutral"]

,episode,name,line,emotion,score
3,4x02 - The Best Burger in New York,Lily,"Barney, Marshall didn't quit his last soul-sucking corporate job just to go work at a bank. He's gonna be an environmental lawyer.\n",neutral,0.653542
4,4x02 - The Best Burger in New York,Ted,"So, what do you guys want to do for dinner?\n",neutral,0.814583
5,4x02 - The Best Burger in New York,Robin,"Great, we'll take five of those.\n",neutral,0.950985
8,4x02 - The Best Burger in New York,Marshall,I thought you started that yesterday.\n,neutral,0.665153
9,4x02 - The Best Burger in New York,Robin,"I finished early, OK?\n",neutral,0.620877
...,...,...,...,...,...
23999,2x21 - Something Borrowed,Lily,I do.\n,neutral,0.899836
24002,2x21 - Something Borrowed,Lily,No.\n,neutral,0.876918
24006,2x21 - Something Borrowed,Marshall,"So where do you want to do it for the first time as a married couple, nice hotel room or a reception hall bathroom?\n",neutral,0.833736
24007,2x21 - Something Borrowed,Lily,"What do you think? Bathroom, of course.\n",neutral,0.503866


In [14]:
df[df["emotion"] == "surprise"]

,episode,name,line,emotion,score
1,4x02 - The Best Burger in New York,Ted,"Okay, first of all, you look like the last pick in the draft. And, second, why are you so excited about some bank?\n",surprise,0.763355
6,4x02 - The Best Burger in New York,Ted,Really? You want to eat here?\n,surprise,0.874170
15,4x02 - The Best Burger in New York,Ted,Chinese?\n,surprise,0.774068
18,4x02 - The Best Burger in New York,Ted,Indian?\n,surprise,0.718132
22,4x02 - The Best Burger in New York,Ted,Mexican?\n,surprise,0.598576
...,...,...,...,...,...
23967,2x21 - Something Borrowed,Marshall,"What? No ""Property of Marshall"" across the back? How are people going to know whose butt that is?\n",surprise,0.477846
23968,2x21 - Something Borrowed,Lily,"What happened? Remember the wedding we wanted, the intimate outdoor ceremony?\n",surprise,0.606214
23971,2x21 - Something Borrowed,Marshall,What?\n,surprise,0.915960
23992,2x21 - Something Borrowed,Robin,"Hey, I found your panties!\n",surprise,0.548242


In [15]:
df[df["emotion"] == "disgust"]

,episode,name,line,emotion,score
7,4x02 - The Best Burger in New York,Robin,"Yeah, I'm freaking starving. I just finished a seven-day cleanse.\n",disgust,0.499789
17,4x02 - The Best Burger in New York,Barney,I don't like Chinese.\n,disgust,0.873254
19,4x02 - The Best Burger in New York,Barney,I just said I don't like Chinese.\n,disgust,0.787071
23,4x02 - The Best Burger in New York,Barney,I just said I don't like Chinese.\n,disgust,0.787071
39,4x02 - The Best Burger in New York,Marshall,"No. Look, the old lady in a bikini is back on. I'm just gonna lie back and get comfortable.\n",disgust,0.609668
...,...,...,...,...,...
23937,2x21 - Something Borrowed,Ted,Marshall accidentally shaved part of his head.\n,disgust,0.607714
23951,2x21 - Something Borrowed,Ted,In a bad way.\n,disgust,0.932019
23960,2x21 - Something Borrowed,Marshall,"Lily, you're not supposed to see me.\n",disgust,0.364336
23974,2x21 - Something Borrowed,Marshall,"Could we even do that? I mean, what about all those people in there?\n",disgust,0.355296


In [16]:
df[df["emotion"] == "anger"]

,episode,name,line,emotion,score
2,4x02 - The Best Burger in New York,Barney,"Our company just bought them out in a ruthless takeover. Took two months. Cost 2,000 jobs. It was brutal. Who wants a T-shirt? Hey, Marshall, they're hiring in the legal department. I could get you a job.\n",anger,0.529787
34,4x02 - The Best Burger in New York,Ted,You are being ridiculous. (\n,anger,0.685565
37,4x02 - The Best Burger in New York,Marshall,"What if I do, Ted. I don't have a switchblade. I don't know how to break-dance and win the begrudging respect of a street g*ng.\n",anger,0.327825
40,4x02 - The Best Burger in New York,Ted,"Go outside, go, go.\n",anger,0.750661
89,4x02 - The Best Burger in New York,Barney,"How lame is free a*t*matic bill pay? How lame is 3.3% APY online savings? Yeah, that's right. Hate to make you look stupid in front of your friends, but you left me no choice.\n",anger,0.570757
...,...,...,...,...,...
23932,2x21 - Something Borrowed,Marshall,"Only Red Andy was falsely accused. Ted, you're my best man! You got to do something!\n",anger,0.733523
23934,2x21 - Something Borrowed,Marshall,No!\n,anger,0.556678
23945,2x21 - Something Borrowed,Ted,It totally does.\n,anger,0.459468
23947,2x21 - Something Borrowed,Barney,"To be honest, I'm, uh, I'm jealous I don't get to wear it.\n",anger,0.295915


In [17]:
df[df["emotion"] == "joy"]

,episode,name,line,emotion,score
0,4x02 - The Best Burger in New York,Barney,"Goliath National Bank. The world leader in credit and banking. God, I love Goliath National Bank!\n",joy,0.863149
11,4x02 - The Best Burger in New York,Lily,We had sushi last night.\n,joy,0.473605
21,4x02 - The Best Burger in New York,Barney,"Weird meats, funny music, side of rice. Why are we splitting hairs?\n",joy,0.431489
28,4x02 - The Best Burger in New York,Barney,"I love this burger so much, I want to sew my ass shut.\n",joy,0.813677
30,4x02 - The Best Burger in New York,Marshall,"Guys, guys, guys. When you've had the best burger in NY, every other burger tastes like my grandpa's feet. But you guys eat up, enjoy my grandpa's feet.\n",joy,0.589246
...,...,...,...,...,...
23977,2x21 - Something Borrowed,Lily,I love it.\n,joy,0.977760
23982,2x21 - Something Borrowed,Barney,"Thank you all for coming. For those of you who don't know me... I'm not the biggest believer in marriage. But... you two are so great together, you know? It's like you were, uh, made for each other.\n",joy,0.463053
23988,2x21 - Something Borrowed,Marshall,"Okay, I'll go first. Lily, there are a million reasons why I love you. You make me laugh and you take care of me when I'm sick. You're sweet, caring and you even created an egg dish and named it after me. She puts a little Italian dressing in scrambled eggs before she cooks them. It's called ""Eggs Marshall,"" and it's awesome. But the main reason that I love you is that you're my best friend, Lily. You're, uh... you're the best friend I ever had. I'm sorry, buddy.\n",joy,0.310599
23991,2x21 - Something Borrowed,Lily,"My turn. Oh, thank you. Marshall, I love you because you're funny and you make me feel loved and you make me feel safe and for our anniversary you gave me a sweatshirt that says, ""Lily and Marshall. Rockin' It Since '96."" I kinda wish I was wearing it right now 'cause it smells like you. But the main reason I love you, Marshall Ericksen, is you make me happy. You make me happy all the time.\n",joy,0.918160


In [18]:
df[df["emotion"] == "sadness"]

,episode,name,line,emotion,score
77,4x02 - The Best Burger in New York,Marshall,"That's where my story ends. Now I'm doomed to walk the earth forever searching for that green door and that red neon sign that says ""Burger.""\n",sadness,0.664379
109,4x02 - The Best Burger in New York,Lily,"Marshall's not doing so well, guys. He really needs to get a job.\n",sadness,0.704286
135,4x02 - The Best Burger in New York,Barney,"Of course. I'm sorry, I forgot to call you. That's not the place. The real place is on 106th and Manhattan Avenue. We're headed there right now.\n",sadness,0.449380
166,4x02 - The Best Burger in New York,Robin,They're unopened.\n,sadness,0.398610
170,4x02 - The Best Burger in New York,Lily,I'm sorry you didn't get your burger.\n,sadness,0.719960
...,...,...,...,...,...
23969,2x21 - Something Borrowed,Marshall,I wish we could have that wedding.\n,sadness,0.858520
23983,2x21 - Something Borrowed,Robin,He's gonna cry.\n,sadness,0.852079
24004,2x21 - Something Borrowed,Marshall,How do you feel?\n,sadness,0.449959
24005,2x21 - Something Borrowed,Lily,Tired. I got married twice today.\n,sadness,0.796927


In [19]:
df[df["emotion"] == "fear"]

,episode,name,line,emotion,score
25,4x02 - The Best Burger in New York,Robin,"Of course, mine comes last. Go ahead, start without me.\n",fear,0.533835
32,4x02 - The Best Burger in New York,Marshall,"It was eight years ago, my first week in New York, and for a kid from Minnesota, the big city was a scary place.\n",fear,0.961926
36,4x02 - The Best Burger in New York,Ted,"Marshall, you have to get over this paranoia. You are not gonna get mugged.\n",fear,0.479497
53,4x02 - The Best Burger in New York,Marshall,(Raising hands in the sky.)\n,fear,0.449196
93,4x02 - The Best Burger in New York,Robin,(On nerves)\n,fear,0.463769
...,...,...,...,...,...
23867,2x21 - Something Borrowed,Lily,Oh. I'm worried my cousin's going to cut it too short.\n,fear,0.729308
23927,2x21 - Something Borrowed,Ted,"Okay, we have a bit of a situation. Let's not panic. Let's just find a solution.\n",fear,0.840826
23933,2x21 - Something Borrowed,Ted,"Okay, all right, come here. Just breathe, breathe, all right? Don't worry. Don't worry. I'll just... I'll take these.\n",fear,0.709433
23948,2x21 - Something Borrowed,Marshall,"Okay, problem solved. Crisis averted. Let's get me married. It looks terrible, doesn't it?\n",fear,0.854302


In [20]:
df.groupby("emotion")["score"].mean()

emotion
anger       0.571104
disgust     0.609106
fear        0.638209
joy         0.701732
neutral     0.697567
sadness     0.651597
surprise    0.696932
Name: score, dtype: float64

In [21]:
df.to_csv("files/himym_dialogues_emotions.csv", index=False)